In [40]:
import pandas as pd
import numpy as np
from sklearn.ensemble import (RandomForestClassifier, 
                              GradientBoostingClassifier, 
                              HistGradientBoostingClassifier,
                              StackingClassifier,
                              IsolationForest)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

In [41]:
# Load Data
train = pd.read_csv('/kaggle/input/competitions/cadde-2026-engine-anomaly-detection/train.csv')
labels = pd.read_csv("/kaggle/input/competitions/cadde-2026-engine-anomaly-detection/train_labels.csv")
test   = pd.read_csv("/kaggle/input/competitions/cadde-2026-engine-anomaly-detection/test.csv")

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Class balance — Normal: {(labels['Anomaly']==0).sum()}, Anomaly: {(labels['Anomaly']==1).sum()}")

Train: (120, 5), Test: (263, 5)
Class balance — Normal: 100, Anomaly: 20


In [42]:
'''
Feature Engineering
The anomaly description says it arises from interactions between features, not single-variable thresholds. 
So we need to create features that capture the physical relationships between sensors.
'''
def add_features(df):
    df = df.copy()
 
    # Physics-based features
    # How much pressure the pump adds (the "work" it does)
    df['Pressure_Diff']    = df['Outlet_Pressure'] - df['Inlet_Pressure']
    # Ratio of outlet to inlet (efficiency indicator)
    df['Pressure_Ratio']   = df['Outlet_Pressure'] / df['Inlet_Pressure']
    # How much flow per unit of pump pressure (efficiency)
    df['Pump_Efficiency']  = df['Flowrate'] / df['Pump_Pressure']
    # How much flow per unit of pressure added
    df['Flow_per_PressureDiff'] = df['Flowrate'] / (df['Outlet_Pressure'] - df['Inlet_Pressure'])
 
    # Interaction features
    df['Temp_x_Flow']      = df['Temperature'] * df['Flowrate']
    df['Temp_x_Pressure']  = df['Temperature'] * df['Pump_Pressure']

    # Polynomial
    df['Temp_sq'] = df['Temperature'] ** 2
    df['Flow_sq'] = df['Flowrate'] ** 2

    # Log transforms to normalize skewness
    df['Log_Flowrate'] = np.log1p(df['Flowrate'])
    df['Log_Temp'] = np.log1p(df['Temperature'])
 
    return df
 
train_fe = add_features(train)
test_fe  = add_features(test)

In [43]:
# Separate features and target
X = train_fe
y = labels['Anomaly']
X_test = test_fe
print(f"\nFeatures used ({X.shape[1]} total): {list(X.columns)}")


Features used (15 total): ['Temperature', 'Pump_Pressure', 'Inlet_Pressure', 'Outlet_Pressure', 'Flowrate', 'Pressure_Diff', 'Pressure_Ratio', 'Pump_Efficiency', 'Flow_per_PressureDiff', 'Temp_x_Flow', 'Temp_x_Pressure', 'Temp_sq', 'Flow_sq', 'Log_Flowrate', 'Log_Temp']


In [44]:
# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

In [45]:
# Add an anomaly score as a feature
iso = IsolationForest(contamination=0.17, random_state=42)
X['Iso_Score'] = iso.fit_predict(X_scaled)
X['Iso_Decision'] = iso.decision_function(X_scaled)
X_test['Iso_Score'] = iso.predict(X_test_scaled)
X_test['Iso_Decision'] = iso.decision_function(X_test_scaled)

In [46]:
# Re-scale with new features
scaler_final = StandardScaler()
X_final = scaler_final.fit_transform(X)
X_test_final = scaler_final.transform(X_test)

In [47]:
# Ensemble
# Base models with constraints to prevent overfitting on tiny data
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)),
    ('hgb', HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=3, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5
)

In [48]:
# Cross-validation

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scores = cross_val_score(stacking_model, X_final, labels['Anomaly'], cv=cv, scoring='roc_auc')

print(f"Ensemble CV AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

Ensemble CV AUC: 0.9825 (+/- 0.0253)


In [49]:
# Train final model
stacking_model.fit(X_final, labels['Anomaly'])
test_probs = stacking_model.predict_proba(X_test_final)[:, 1]

In [50]:
# Submission
submission = pd.DataFrame({"id": np.arange(len(test_probs)), "pred": test_probs})
submission.to_csv('submission.csv', index=False)